# IBM Model 1 を実装して対訳から単語アライメントを学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/ibm_model1_scratch.ipynb)

NLP クックブック B-21 ／ 仕組みを理解するための実装

日英の対訳文ペアから、辞書なしで単語同士の対応（単語アライメント）を見つけたい。EM アルゴリズム（E ステップの事後確率計算と M ステップの翻訳確率 $t(f \mid e)$ の更新）を自分で実装し、対数尤度が反復ごとに単調増加することとアライメント行列の獲得を確かめる。

In [1]:
import math
from collections import defaultdict
import numpy as np
import pandas as pd

pairs = [
    (["猫", "が", "魚", "を", "食べる"], ["the", "cat", "eats", "fish"]),
    (["犬", "が", "肉", "を", "食べる"], ["the", "dog", "eats", "meat"]),
    (["猫", "は", "小さい"], ["the", "cat", "is", "small"]),
    (["犬", "は", "大きい"], ["the", "dog", "is", "big"]),
    (["小さい", "猫", "が", "走る"], ["the", "small", "cat", "runs"]),
    (["大きい", "犬", "が", "走る"], ["the", "big", "dog", "runs"]),
    (["鳥", "が", "空", "を", "飛ぶ"], ["the", "bird", "flies", "in", "the", "sky"]),
    (["小さい", "鳥", "が", "飛ぶ"], ["a", "small", "bird", "flies"]),
    (["魚", "が", "水", "を", "泳ぐ"], ["the", "fish", "swims", "in", "water"]),
    (["赤い", "リンゴ", "を", "食べる"], ["eat", "a", "red", "apple"]),
    (["甘い", "リンゴ", "を", "買う"], ["buy", "a", "sweet", "apple"]),
    (["青い", "空", "を", "見る"], ["see", "the", "blue", "sky"]),
    (["青い", "海", "を", "見る"], ["see", "the", "blue", "sea"]),
    (["子ども", "が", "本", "を", "読む"], ["the", "child", "reads", "a", "book"]),
    (["学生", "が", "本", "を", "買う"], ["the", "student", "buys", "a", "book"]),
    (["先生", "が", "本", "を", "読む"], ["the", "teacher", "reads", "a", "book"]),
    (["学生", "が", "学校", "で", "学ぶ"], ["the", "student", "studies", "at", "school"]),
    (["先生", "が", "学校", "で", "教える"], ["the", "teacher", "teaches", "at", "school"]),
    (["猫", "が", "ベッド", "で", "寝る"], ["the", "cat", "sleeps", "on", "the", "bed"]),
    (["犬", "が", "ベッド", "で", "寝る"], ["the", "dog", "sleeps", "on", "the", "bed"]),
    (["子ども", "が", "学校", "で", "走る"], ["the", "child", "runs", "at", "school"]),
    (["私", "は", "赤い", "花", "を", "見る"], ["i", "see", "a", "red", "flower"]),
    (["私", "は", "甘い", "ケーキ", "を", "食べる"], ["i", "eat", "a", "sweet", "cake"]),
    (["大きい", "魚", "が", "泳ぐ"], ["a", "big", "fish", "swims"]),
]
print("対訳文ペア数:", len(pairs))

対訳文ペア数: 24


In [2]:
def train_ibm_model1(sentence_pairs, n_iter=15):
    """EM アルゴリズムで単語翻訳確率 t(f | e) を推定し、各反復の対数尤度を返す"""
    f_vocab = sorted({f for _, fs in sentence_pairs for f in fs})
    e_vocab = sorted({e for es, _ in sentence_pairs for e in es} | {"NULL"})
    t = {(f, e): 1.0 / len(f_vocab) for f in f_vocab for e in e_vocab}
    log_likelihoods = []

    for _ in range(n_iter):
        ll = 0.0
        count, total_e = defaultdict(float), defaultdict(float)
        for es, fs in sentence_pairs:
            es_null = ["NULL"] + es
            norm_const = -math.log(len(es_null))
            for f in fs:
                s = sum(t[(f, e)] for e in es_null)
                ll += norm_const + math.log(s)
                for e in es_null:
                    post = t[(f, e)] / s           # E ステップ: 事後アライメント確率
                    count[(f, e)] += post
                    total_e[e] += post
        log_likelihoods.append(ll)
        for k in t:                                # M ステップ: 期待頻度から確率を更新
            t[k] = count[k] / total_e[k[1]] if total_e[k[1]] > 0 else 0.0
    return t, log_likelihoods

t_table, lls = train_ibm_model1(pairs, n_iter=15)
print("対数尤度の推移:", [round(x, 2) for x in lls[:5]] + ["..."] + [round(x, 2) for x in lls[-3:]])
print("単調増加の確認:", all(lls[i] <= lls[i + 1] + 1e-9 for i in range(len(lls) - 1)))

print("\n=== 学習された翻訳確率 t(英語 | 日本語) の上位 ===")
for ja_w in ["猫", "犬", "食べる", "赤い", "学校", "本"]:
    top = sorted(((p, f) for (f, e), p in t_table.items() if e == ja_w and p > 0.1), reverse=True)[:3]
    print(f"  {ja_w:4s} → {[(f, round(p, 3)) for p, f in top]}")

対数尤度の推移: [-412.21, -244.22, -225.86, -212.23, -203.99, '...', -193.56, -193.43, -193.34]
単調増加の確認: True

=== 学習された翻訳確率 t(英語 | 日本語) の上位 ===
  猫    → [('cat', 0.994)]
  犬    → [('dog', 0.993)]
  食べる  → [('eats', 0.513), ('eat', 0.486)]
  赤い   → [('red', 0.933)]
  学校   → [('school', 0.5), ('at', 0.5)]
  本    → [('book', 0.94)]


In [3]:
# 学習済みモデルで文対の単語アライメント行列と Viterbi アライメントを求める
es, fs = ["大きい", "猫", "が", "魚", "を", "食べる"], ["the", "big", "cat", "eats", "fish"]
es_null = ["NULL"] + es
mat = pd.DataFrame(
    [[t_table.get((f, e), 0.0) for e in es_null] for f in fs],
    index=fs, columns=es_null,
)
print("=== 翻訳確率行列 t(f | e) ===")
print(mat.round(2).to_string())

print("\n=== Viterbi アライメント (各英単語に最も確からしい日本語語彙) ===")
for f in fs:
    best_e = mat.loc[f].idxmax()
    print(f"  {f:5s} ← {best_e} ({mat.loc[f, best_e]:.3f})")

=== 翻訳確率行列 t(f | e) ===
      NULL   大きい     猫     が     魚     を   食べる
the   0.87  0.00  0.01  0.94  0.00  0.17  0.00
big   0.00  0.99  0.00  0.00  0.00  0.00  0.00
cat   0.00  0.00  0.99  0.00  0.00  0.00  0.00
eats  0.00  0.00  0.00  0.00  0.00  0.00  0.51
fish  0.00  0.00  0.00  0.00  0.97  0.00  0.00

=== Viterbi アライメント (各英単語に最も確からしい日本語語彙) ===
  the   ← が (0.941)
  big   ← 大きい (0.995)
  cat   ← 猫 (0.994)
  eats  ← 食べる (0.513)
  fish  ← 魚 (0.972)


- IBM Model 1 の目的関数（対数尤度）はパラメータに関して凸（厳密に言えば確率分布上で凹）であるため、EM アルゴリズムは局所解に陥らず、反復ごとに必ず単調増加して大域最適解へ収束する。
- 日本語の助詞（「が」「を」）や英語の冠詞 (`the`, `a`) は相手側に対応する語がないことも多く、`NULL` や共起頻度の高い助詞へ確率が分散する（上の例でも *the* は「が」や `NULL` に割り当てられる）。
- 語順を考慮しないため、長い文では遠い位置への誤った対応が起きやすい。実務の統計的機械翻訳では IBM Model 1 の結果を初期値にして位置モデルを含む HMM や IBM Model 4（`fast_align` など）へ進み、日→英と英→日の双方向アライメントを対称化して使う（機械翻訳の章の「Moses で統計的機械翻訳システムを作る」を参照）。